# Stage 5: Non-Destructive Folder Materialization

## Overview and Purpose
Once visual character clusters are established, the images must be organized into a clean, human-navigable directory structure. In many naive workflows, files are moved or renamed in place, creating irreversible risks of data corruption or loss.

This notebook executes **Stage 5** with a strict **non-destructive architecture**:
1. **Source Preservation**: Original source files are strictly read-only and never modified, moved, or deleted.
2. **Materialization Modes**:
   - `hardlink` (default): Instantaneous file creation using filesystem hard links (zero extra disk space, points to existing inode).
   - `copy`: Full independent copies with timestamp and metadata preservation (`shutil.copy2`), safe across distinct partitions/drives.
   - `symlink`: Symbolic links pointing to source paths.
3. **Collision Prevention**: Every destination file is assigned a deterministic index prefix (`{embedding_row:07d}__{stem}.ext`), preventing collisions when source images share identical filenames across different source directories.
4. **Folder Metadata & Manifests**: Every destination folder receives an index manifest (`_folder_manifest.csv`), metadata JSON (`_cluster_info.json`), and contact sheet (`_contact_sheet.jpg`).
5. **Post-Materialization Integrity Validation**: Automatically verifies that 100% of planned files exist and have byte sizes matching the original sources.

## 1. Imports and Environment Setup
Import materialization operations, planner, and validation tools.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

from anime_character_organizer import (
    MaterializationConfig,
    run_folder_materialization,
)
from anime_character_organizer.materialization.planner import build_materialization_plan
from anime_character_organizer.materialization.validation import validate_materialized_files
from anime_character_organizer.utils.runs import find_latest_run

## 2. Materialization Configuration
Configure output directory, linking mode, and fallback policies.

In [ ]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_CLUSTER_RUN = None  # Auto-discovers latest 04_hdbscan_clustering run

# Materialization mode: 'hardlink' (0 disk space), 'copy', or 'symlink'
MATERIALIZATION_MODE = "hardlink"
ALLOW_FALLBACK_TO_COPY = True

# Safety policy on existing files: 'error', 'skip', or 'overwrite'
ON_EXISTING = "error"

# Custom destination folder (None places output under PROJECT_DIR/organized_output/anime_organized_<timestamp>)
FINAL_OUTPUT_DIR = None

print("Materialization Settings:")
print(f"  Mode:                  {MATERIALIZATION_MODE}")
print(f"  Fallback to Copy:      {ALLOW_FALLBACK_TO_COPY}")
print(f"  On Existing Conflict:  {ON_EXISTING}")

## 3. Folder Materialization Workflow Execution
Run the materialization workflow. Pre-flight checks verify all source files exist before any links or copies are generated.

In [ ]:
results = run_folder_materialization(
    project_dir=PROJECT_DIR,
    previous_run_dir=PREVIOUS_CLUSTER_RUN,
    final_output_dir=FINAL_OUTPUT_DIR,
    materialization_mode=MATERIALIZATION_MODE,
    allow_hardlink_fallback_to_copy=ALLOW_FALLBACK_TO_COPY,
    on_existing=ON_EXISTING,
    show_progress=True,
)

print("Folder materialization completed successfully.")
print(f"Organized Directory: {results['final_output_dir']}")

## 4. Operation Summary & Top Materialized Folders
Review how many files were hardlinked, copied, or skipped, and inspect the largest character folders created.

In [ ]:
print("Operation Summary:")
display(results["operation_counts_df"])

print("Top 25 Organized Character Folders:")
display(results["folder_counts_df"].head(25))

## 5. Post-Materialization Integrity Validation
Verify that every destination file exists, is accessible as a regular file or link, and matches the byte size of its corresponding source image.

In [ ]:
val_df = results["validation_df"]
all_valid = bool(val_df["is_valid"].all())
print(f"All destination files verified valid and matching source: {all_valid}")
display(val_df["is_valid"].value_counts().to_frame(name="File Count"))

if not all_valid:
    display(val_df[~val_df["is_valid"]].head(25))

## 6. Assumptions, Limitations, and Next Steps

### Assumptions & Limitations
- **Hardlink Filesystem Boundary**: Hard links require the source files and the output directory to reside on the same filesystem / mount point. If they reside on different partitions or filesystems, `allow_hardlink_fallback_to_copy=True` automatically falls back to copy mode with zero data loss.
- **Disk Space**: Hard links consume zero additional disk storage (they share the existing inode). Full copies will require additional storage equal to the sum of the organized images.

### Next Steps
The materialized directory structure is ready for manual inspection, or can be passed to **Stage 6 (`06_cluster_naming_with_anime_tagger.ipynb`)** for automatic semantic character name inference.